## This notebook enables you to run the local Qwen3-8B model on Kaggle's GPU free of charge. You can casually chat with the model and conduct simple experiments.

In [ ]:
!pip install bitsandbytes

In [3]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="Qwen/Qwen3-8B", local_dir='local_checkpoints/qwen')

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

'/home/tasin/Documents/kaggle_ml/local_checkpoints/qwen'

In [1]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    'local_checkpoints/qwen',
    quantization_config = quantization_config,
    device_map = 'cuda:0',
    dtype = torch.float16,
    local_files_only=True)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    'local_checkpoints/qwen',
    local_files_only=True
)

In [4]:
messages = [
    {"role": "system", "content": "You are a chatbot that provides clear, concise, short answer that can be understood by everybody."},
]

In [5]:
MAX_MESSAGES = 5 
while True:
    user_input = input("You: ")
    if user_input.lower() in ["bye", "quit", "exit"]:
        break

    messages.append({"role": "user", "content": user_input})
    messages = [messages[0]] + messages[1:][-5:]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate response
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    response = tokenizer.decode(output_ids[0:], skip_special_tokens=True).strip("\n")
    print(f"Qwen: {response}")
    messages.append({"role": "assistant", "content": response})

You:  Hello


/home/tasin/anaconda3/envs/ml/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Qwen: Hello! How can I assist you today?


You:  Tell me about Honda.


Qwen: Honda is a Japanese multinational automotive manufacturer known for producing cars, motorcycles, and power equipment. Founded in 1946, it is one of the world's largest automakers. Some of its popular car models include the Civic, CR-V, and Accord. Honda is also known for its innovation in hybrid and electric vehicle technology.


You:  Who is the founder?


Qwen: Honda was founded by **Soichiro Honda** in 1946. He was a visionary engineer and played a key role in shaping the company into a global automotive leader.


You:  bye
